In [3]:
import pandas as pd

url = "https://orca.casa.ucl.ac.uk/~jreades/data/20260917-London-listings.csv.gz"

listings = pd.read_csv(
    url,
    usecols=["id", "host_id", "room_type"],
    dtype={"id": "string", "host_id": "string"},
)

assert listings[["id", "host_id"]].notna().all().all(), "Missing listing or host IDs"
assert listings["id"].is_unique, "Duplicate listing IDs"

print(f"Listings: {len(listings):,}")
print(f"Host accounts: {listings['host_id'].nunique():,}")

Listings: 91,954
Host accounts: 51,348


In [4]:
full_counts = listings.groupby("host_id")["id"].nunique()

entire_listings = listings.loc[listings["room_type"].eq("Entire home/apt")]
entire_counts = entire_listings.groupby("host_id")["id"].nunique()

print("Full sample:", len(full_counts), full_counts.sum())
print("Entire homes/apartments:", len(entire_counts), entire_counts.sum())

Full sample: 51348 91954
Entire homes/apartments: 34055 61074


In [5]:
def group_summary(counts, selected):
    group = counts.loc[selected]
    number_of_hosts = len(group)
    number_of_listings = int(group.sum())

    return {
        "Host Count": number_of_hosts,
        "Share of Hosts": number_of_hosts / len(counts),
        "Listing Count": number_of_listings,
        "Share of Listings": number_of_listings / counts.sum(),
    }

baseline = pd.DataFrame([
    {"Host Group": "1 listing", **group_summary(full_counts, full_counts.eq(1))},
    {"Host Group": "2+ listings", **group_summary(full_counts, full_counts.ge(2))},
])

formats = {
    "Host Count": "{:,}".format,
    "Share of Hosts": "{:.2%}".format,
    "Listing Count": "{:,}".format,
    "Share of Listings": "{:.2%}".format,
}

print(baseline.to_string(index=False, formatters=formats))

 Host Group Host Count Share of Hosts Listing Count Share of Listings
  1 listing     40,964         79.78%        40,964            44.55%
2+ listings     10,384         20.22%        50,990            55.45%


In [6]:
samples = {
    "Full sample": full_counts,
    "Entire homes/apartments": entire_counts,
}

rows = []
for cutoff in (2, 3):
    for sample, counts in samples.items():
        rows.append({
            "Sample": sample,
            "Threshold": f"{cutoff}+ listings",
            **group_summary(counts, counts.ge(cutoff)),
        })

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False, formatters=formats))

                 Sample   Threshold Host Count Share of Hosts Listing Count Share of Listings
            Full sample 2+ listings     10,384         20.22%        50,990            55.45%
Entire homes/apartments 2+ listings      5,161         15.15%        32,180            52.69%
            Full sample 3+ listings      4,939          9.62%        40,100            43.61%
Entire homes/apartments 3+ listings      2,765          8.12%        27,388            44.84%


### Sensitivity to the listing threshold

In the September 2026 London sample, host accounts with at least two entire-home or apartment listings managed **52.69%** of the **61,074** listings in this category. Raising the cutoff to three moves **2,396 accounts** with exactly two listings each out of the qualifying group. Together, those accounts managed **4,792 listings**.

Both percentages use all **61,074** listings in the category as the denominator. The qualifying group's share therefore falls by **7.85 percentage points**, from **52.69%** to **44.84%**. The 2+ group manages just over half of the listings, while the 3+ group manages just under half. Whether the group manages a majority depends on the cutoff.